In [ ]:
import torch
import pandas as pd
from epiweeks import Week
import preprocess_data as prep
import model as mc
import matplotlib.pyplot as plt 


pd.options.mode.chained_assignment = None

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

regioes_estados = {
        'Sul': ['SC', 'PR', 'RS'],
        'Sudeste': ['SP', 'MG', 'RJ', 'ES'],
        'Nordeste': ['BA', 'CE', 'PE', 'PB', 'PI', 'RN', 'MA', 'AL', 'SE'],
        'Centro-Oeste': ['DF', 'MT', 'MS', 'GO'],
        'Norte': ['RO', 'AC', 'AM', 'RR', 'PA', 'AP', 'TO']
    } 
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

columns_to_normalize = ['casos','epiweek', 'biome']

predict_n = 36
max_epiweek = 16
    
boxcox = False

TEST_YEAR = 2023


In [ ]:
batch_size = 2
epochs = 250
media =True
verbose = 1
doenca = 'chikungunya'
min_delta = 0.0
patience= 25
lr = 2e-4
min_year = 2015
model_name = f'dengue_base'

In [3]:
df = prep.load_cases_data(filename= f'./data/{doenca}.csv.gz')

enso = None
enso_neutro = None

In [4]:
for region in regioes_estados.keys(): 

    print(region)

    label = f'{region}_{TEST_YEAR-1}_{model_name}'

    df_reg = df.loc[df.uf.isin(regioes_estados[region])]
    df_reg = df_reg.loc[df_reg.index >= pd.to_datetime(Week(2015,1).startdate())]


    X_train, y_train, X_future, norm_values, norm_enso =  prep.generate_regional_train_samples(df_reg, enso, 
                                                                                    TEST_YEAR,
                                                                                    max_epiweek = max_epiweek,
                                                                                    columns_to_normalize = columns_to_normalize,
                                                                                    min_year = min_year, 
                                                                                    boxcox = boxcox,
                                                                                    media = media)

    model = mc.LSTMLogNormalModel(hidden=64, features=len(columns_to_normalize) + 1, 
                        predict_n=52 - max_epiweek)
    
    model_path = f'./saved_models/trained_dengue_{region}_{TEST_YEAR-1}_base.pt'
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    model.to(device)  
    
    trained_model, hist = mc.train_model(

            model=model,

            X_train=X_train,

            X_future=X_future,

            Y_train=y_train,

            label=label,

            batch_size=batch_size,

            epochs=epochs,

            lr=lr,

            patience=patience,

            verbose=verbose,

            doenca=doenca
)


Sul
Epoch 1/250 | Train: 0.0459 | Val: 0.0297
Epoch 2/250 | Train: 0.0422 | Val: 0.0278
Epoch 3/250 | Train: 0.0428 | Val: 0.0274
Epoch 4/250 | Train: 0.0406 | Val: 0.0271
Epoch 5/250 | Train: 0.0393 | Val: 0.0269
Epoch 6/250 | Train: 0.0391 | Val: 0.0269
Epoch 7/250 | Train: 0.0374 | Val: 0.0270
Epoch 8/250 | Train: 0.0375 | Val: 0.0270
Epoch 9/250 | Train: 0.0372 | Val: 0.0270
Epoch 10/250 | Train: 0.0350 | Val: 0.0269
Epoch 11/250 | Train: 0.0375 | Val: 0.0272
Epoch 12/250 | Train: 0.0349 | Val: 0.0273
Epoch 13/250 | Train: 0.0343 | Val: 0.0278
Epoch 14/250 | Train: 0.0336 | Val: 0.0277
Epoch 15/250 | Train: 0.0341 | Val: 0.0277
Epoch 16/250 | Train: 0.0336 | Val: 0.0277
Epoch 17/250 | Train: 0.0346 | Val: 0.0276
Epoch 18/250 | Train: 0.0338 | Val: 0.0275
Epoch 19/250 | Train: 0.0336 | Val: 0.0274
Epoch 20/250 | Train: 0.0340 | Val: 0.0275
Epoch 21/250 | Train: 0.0348 | Val: 0.0275
Epoch 22/250 | Train: 0.0333 | Val: 0.0276
Epoch 23/250 | Train: 0.0331 | Val: 0.0279
Epoch 24/250 | T